# Stage 1 — Non-Instruction Fine-Tuning
Adapts the base model to diabetes domain language using raw paragraphs.

**Runtime → Change runtime type → T4 GPU before running.**

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT_DIR = "/content/drive/MyDrive/domain-ai-assistant-finetuning"
os.makedirs(f"{PROJECT_DIR}/data", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/reports", exist_ok=True)
print("Project dir:", PROJECT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project dir: /content/drive/MyDrive/domain-ai-assistant-finetuning


## 2. Install dependencies

In [ ]:
%%capture

!pip uninstall -y trl transformers accelerate datasets pyarrow

!pip install -U "unsloth[colab-new]"

!pip install \
    transformers==4.55.4 \
    trl==0.20.0 \
    accelerate==1.10.1 \
    datasets==3.6.0 \
    pyarrow==20.0.0

## 3. Load raw domain text
Upload `non_instruction_data.txt` into `/content/drive/MyDrive/domain-ai-assistant-finetuning/data/` first.

In [ ]:
data_path = f"{PROJECT_DIR}/data/non_instruction_data.txt"
with open(data_path, encoding="utf-8") as f:
    raw_text = f.read()

paragraphs = [p.strip() for p in raw_text.split("\n\n") if p.strip()]
print(f"Loaded {len(paragraphs)} paragraphs")
assert len(paragraphs) >= 50, f"Only {len(paragraphs)} paragraphs -- need >= 50"


Loaded 106 paragraphs


## 4. Build dataset (raw text, no pre-tokenization)
SFTTrainer tokenizes internally via `dataset_text_field` — we do NOT pre-tokenize here. Pre-tokenizing then removing the text column causes SFTTrainer to silently skip the data.

In [ ]:
from datasets import Dataset

MAX_CHARS = 800

def chunk_paragraphs(paragraphs, max_chars=MAX_CHARS):
    chunks, current = [], ""
    for p in paragraphs:
        if len(current) + len(p) > max_chars and current:
            chunks.append(current.strip())
            current = p
        else:
            current = f"{current} {p}".strip()
    if current:
        chunks.append(current.strip())
    return chunks

chunks = chunk_paragraphs(paragraphs)
print(f"{len(chunks)} training chunks")

# Keep as raw text Dataset -- SFTTrainer handles tokenization
dataset = Dataset.from_dict({"text": chunks})
print(dataset[0]["text"][:200])


46 training chunks
Diabetes, also known as diabetes mellitus, is a disease in which your blood glucose , or blood sugar, levels are too high. Glucose is your body's main source of energy. Your body can make glucose, but


## 5. Load base model with Unsloth (4-bit QLoRA)

In [ ]:
from unsloth import FastLanguageModel
import torch

BASE_MODEL = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("Model loaded with LoRA adapters.")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/unsloth_zoo/__init__.py:410: UserWarning: Unsloth fused-forward install skipped: requires transformers >= 4.56.0.
  _install_fused_forward()


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 4.55.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.7.2 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Model loaded with LoRA adapters.


## 6. Train
`max_steps=300` keeps this under 30 min on T4. Checkpoints save to Drive every 50 steps so a disconnect does not lose progress.

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer
import torch
import traceback

CHECKPOINT_DIR = f"{PROJECT_DIR}/checkpoints/stage1_non_instruction"

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,

    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,

    learning_rate=2e-4,
    warmup_steps=10,
    max_steps=300,

    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),

    logging_steps=10,

    # IMPORTANT: Disable checkpoint saving
    save_strategy="no",

    optim="adamw_8bit",
    seed=42,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=training_args,
)

try:
    trainer.train()
    print("Training completed successfully!")

except Exception:
    traceback.print_exc()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/46 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 46 | Num Epochs = 50 | Total steps = 300
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Step,Training Loss
10,1.667600
20,1.304100
30,0.805800
40,0.345800
50,0.110600
60,0.050700
70,0.033000
80,0.026300
90,0.019000
100,0.013500


Training completed successfully!


## 7. Save adapter to Drive

In [ ]:
SAVE_PATH = f"{PROJECT_DIR}/checkpoints/stage1_non_instruction_final"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Stage 1 model saved successfully!")
print(SAVE_PATH)


Stage 1 model saved successfully!
/content/drive/MyDrive/domain-ai-assistant-finetuning/checkpoints/stage1_non_instruction_final


## 8. Sanity test

In [ ]:
FastLanguageModel.for_inference(model)
PROMPT_TEMPLATE = (
    "Below is an instruction that describes a task. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n### Response:\n"
)

inputs = tokenizer(
    [PROMPT_TEMPLATE.format(instruction="What is diabetes?")],
    return_tensors="pt"
).to("cuda")
output = model.generate(**inputs, max_new_tokens=80)
print(tokenizer.decode(output[0], skip_special_tokens=True))


Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What is diabetes?

### Response:
Diabetes, also known as diabetes mellitus, is a disease in which your blood glucose , or blood sugar, levels are too high. Glucose comes from foods you eat. The cells of your body need glucose for energy. A hormone called insulin helps the glucose get into your cells. Treatments for diabetes can depend on the type. Common treatments include a diabetic meal plan , regular physical activity,


In [ ]:
!ls -lh /content/drive/MyDrive/domain-ai-assistant-finetuning/checkpoints

total 8.0K
drwx------ 3 root root 4.0K Jul  9 07:31 stage1_non_instruction
drwx------ 2 root root 4.0K Jul  9 07:50 stage1_non_instruction_final
